# Section 4: Linear Classifiers

In the previous sections, we explored the K-Nearest Neighbor (K-NN) classifier. While simple, it was fundamentally flawed for image classification due to its $O(N)$ prediction time and reliance on raw pixel distance.

We now introduce the most important building block in deep learning: the **Parametric Approach**, specifically the **Linear Classifier**. Unlike K-NN, which memorizes the entire dataset, a parametric model summarizes the knowledge of the training data into a set of parameters (weights). Once trained, the training data can be discarded!

## 4.1 The Parametric Approach Overview

In a parametric model, we define a function $f(\mathbf{x}, \mathbf{W})$ that takes the input image $\mathbf{x}$ and a set of parameters $\mathbf{W}$ (weights) and outputs the predicted class scores.

The simplest possible function is a linear mapping:

$$ f(\mathbf{x}, \mathbf{W}) = \mathbf{W}\mathbf{x} + \mathbf{b} $$

Where:
*   $\mathbf{x}$ is the input image tensor flattened into a single column vector.
*   $\mathbf{W}$ is the weight matrix (the parameters we will eventually learn).
*   $\mathbf{b}$ is the bias vector. It allows the function to shift the output scores independently of the input data $\mathbf{x}$ (e.g., if cats are much more common in our dataset, the cat bias might be automatically learned to be higher).

## 4.2 The Algebraic Viewpoint

Let's break down the exact dimensions of this matrix multiplication for the CIFAR-10 dataset (which contains $32 \times 32 \times 3$ images and 10 classes).

1.  **Flatten the Image:** The input image $\mathbf{x}$ is originally $32 \times 32 \times 3$. We flatten it into a single column vector of size $3072 \times 1$ (since $32 \times 32 \times 3 = 3072$).
2.  **Output Scores:** We want 10 scores (one for each class). Thus, our output $f(\mathbf{x}, \mathbf{W})$ must be a $10 \times 1$ vector.
3.  **The Weight Matrix:** By the rules of matrix multiplication, for $\mathbf{W}\mathbf{x}$ to produce a $10 \times 1$ vector when $\mathbf{x}$ is $3072 \times 1$, the matrix $\mathbf{W}$ must be **$10 \times 3072$**.
4.  **The Bias Vector:** The bias $\mathbf{b}$ is simply added to the output, so it must also be **$10 \times 1$**.

$$ \underbrace{f(\mathbf{x}, \mathbf{W})}_{10 \times 1} = \underbrace{\mathbf{W}}_{10 \times 3072} \cdot \underbrace{\mathbf{x}}_{3072 \times 1} + \underbrace{\mathbf{b}}_{10 \times 1} $$

> [!NOTE]
> Notice the incredible efficiency here! To predict the class of a new image, we only perform a single matrix multiplication and an addition. The prediction time is $O(1)$ relative to the size of the training dataset. This solves the fatal flaw of K-NN.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

# --- NUMPY IMPLEMENTATION ---
# Let's simulate a linear classifier pass for a single CIFAR-10 image

# 1. Simulate a flattened image (3072 pixels)
x = np.random.randn(3072, 1) 

# 2. Simulate random weights (10 classes x 3072 pixels)
W = np.random.randn(10, 3072)

# 3. Simulate random biases (10 classes)
b = np.random.randn(10, 1)

# 4. Compute the linear function: Wx + b
scores = W.dot(x) + b

print("--- NUMPY ---")
print("Shape of x:", x.shape)
print("Shape of W:", W.shape)
print("Shape of b:", b.shape)
print("Shape of output scores:", scores.shape)
print("Raw scores for the 10 classes:\n", scores.flatten())

# --- PYTORCH EQUIVALENT ---
print("\n--- PYTORCH ---")
# In PyTorch, nn.Linear automatically handles the Wx + b computation.
# Note: PyTorch linear layers expect inputs of shape (batch_size, features).
# So underneath, it computes: xW^T + b (transposed weight matrix).

linear_classifier = nn.Linear(in_features=3072, out_features=10)
x_torch = torch.randn(1, 3072) # Batch of 1 image

# Forward pass
scores_torch = linear_classifier(x_torch)

print(f"PyTorch input shape: {x_torch.shape}")
print(f"PyTorch output shape: {scores_torch.shape}")

## 4.3 The Visual Viewpoint: Class Templates

If we examine the matrix $\mathbf{W}$ closely, we notice it has 10 rows (one for each class), and each row has 3072 elements (exactly the size of a flattened image).

We can take a single row of $\mathbf{W}$, say the row corresponding to the "car" class, and **un-flatten** it back into a $32 \times 32 \times 3$ image. 

**What does this look like?**
Because the score for the "car" class is generated by taking the dot product between the "car row" of $\mathbf{W}$ and the input image $\mathbf{x}$, the "car row" effectively acts as a **template** (or a matched filter) for cars. The linear classifier is simply comparing the input image to 10 learned templates simultaneously via the dot product.

When trained, these templates begin to look like blurred, generalized versions of their respective classes. A trained car template might look like a red blob sitting on top of two black circles (wheels), because that pattern yields a high dot product with actual images of cars.

## 4.4 The Geometric Viewpoint: Hyperplanes

Geometrically, we can imagine the input images existing as points in a high-dimensional space (e.g., 3072 dimensions). 

Each row of the weight matrix $\mathbf{W}$ defines a hyperplane (a flat, multidimensional boundary) in this space. The linear classifier attempts to draw 10 different hyperplanes that carve up the space, separating the cats from the dogs, the ships from the planes, etc.

*   The orientation of the hyperplane is controlled by $\mathbf{W}$.
*   The position (offset from the origin) is controlled by the bias $\mathbf{b}$. Without $\mathbf{b}$, all hyperplanes would be forced to perfectly intersect the origin $(0,0,0...)$, severely restricting the classifier's flexibility.

### The Bias Trick
Sometimes, to simplify the math in proofs or code, we use the "bias trick". We append a constant $1$ to the input vector $\mathbf{x}$, making it $3073 \times 1$. We then absorb the bias $\mathbf{b}$ into the weight matrix $\mathbf{W}$ as an extra column, making it $10 \times 3073$.

This allows us to write the linear function purely as a single matrix multiplication:
$$ f(\mathbf{x}, \mathbf{W}) = \mathbf{W}\mathbf{x} $$

## 4.5 Hard Cases for Linear Classifiers

While powerful and fast, linear classifiers are inherently limited because they can only draw *straight* lines (hyperplanes). They will fail completely if the data is not linearly separable.

Three classic hard cases:
1.  **Parity / XOR Problem:** Class 1 exists in quadrants I and III, while Class 2 exists in quadrants II and IV. No single straight line can separate them.
2.  **Circular / Ring Data:** Class 1 is clustered in the center, and Class 2 forms a ring around it. 
3.  **Multimodal Distributions:** A class (like horses) consists of two distinct visual modes (e.g., horses facing left vs horses facing right). A linear classifier can only learn *one* template per class. If it tries to merge a left-facing horse and a right-facing horse, it creates a useless two-headed horse template.

To solve these hard cases, we will eventually need the non-linear capabilities of Deep Neural Networks.

---
### Summary of Section 4
*   **Concepts Introduced:** Parametric approach, Linear Classifier, Bias Trick, Algebraic viewpoint (dimensions), Visual viewpoint (templates), Geometric viewpoint (hyperplanes), Hard Cases (XOR, Multimodal).
*   **Equations Derived:** $f(\mathbf{x}, \mathbf{W}) = \mathbf{W}\mathbf{x} + \mathbf{b}$
*   **Notation Introduced:** $\mathbf{W}$ (Weight matrix), $\mathbf{b}$ (Bias vector), $f(\mathbf{x}, \mathbf{W})$ (Scoring function).
*   **Dependencies for Next Section:** We now have a mechanism that takes an image and outputs 10 arbitrary scores. We need a mathematical way to evaluate how "good" or "bad" these scores are, which brings us to Loss Functions.